In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "20"
os.environ["MKL_NUM_THREADS"] = "20"

import torch
torch.set_num_threads(20)

import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [2]:
class Stage1Dataset(torch.utils.data.Dataset):

    def __init__(self, root, transform=None):

        self.samples = []
        self.transform = transform
        self.classes = ["Normal", "Abnormal"]

        for label, cls in enumerate(self.classes):

            folder = os.path.join(root, cls)

            for f in os.listdir(folder):

                if f.lower().endswith((".jpg",".png",".jpeg")):
                    self.samples.append((os.path.join(folder,f),label))


    def __len__(self):
        return len(self.samples)


    def __getitem__(self, idx):

        path,label = self.samples[idx]

        img = Image.open(path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img,label

In [3]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

In [4]:
train_root = r"C:\Users\amarh\Desktop\My Professionals\Projects\CCTV Crime\Datasets\BALANCED_200K"
val_root   = r"C:\Users\amarh\Desktop\My Professionals\Projects\CCTV Crime\Datasets\Test"

train_ds = Stage1Dataset(train_root, train_transform)
val_ds   = Stage1Dataset(val_root, val_transform)

print("Train size:", len(train_ds))
print("Val size:", len(val_ds))

Train size: 200000
Val size: 111308


In [5]:
train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=32,
    shuffle=False,
)

In [6]:
mobilenet = models.mobilenet_v2(weights="IMAGENET1K_V1")

# freeze early layers
for p in mobilenet.features[:-4].parameters():
    p.requires_grad = False

mobilenet.classifier = nn.Identity()

class Stage1Model(nn.Module):

    def __init__(self):
        super().__init__()

        self.backbone = mobilenet
        self.head = nn.Linear(1280,2)

    def forward(self,x):

        x = self.backbone(x)
        return self.head(x)


model = Stage1Model().to(device)

print("Model ready")

Model ready


In [7]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0003
)

In [8]:
EPOCHS = 12

for epoch in range(EPOCHS):

    model.train()

    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for imgs,labels in pbar:

        imgs = imgs.to(device)
        labels = labels.to(device)

        logits = model(imgs)

        loss = criterion(logits,labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        preds = torch.argmax(logits,dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix(
            loss = loss.item(),
            train_acc = correct/total
        )

    train_acc = correct / total


    # validation
    model.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for imgs,labels in val_loader:

            imgs = imgs.to(device)
            labels = labels.to(device)

            logits = model(imgs)

            preds = torch.argmax(logits,dim=1)

            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total

    print(f"\nEpoch {epoch+1} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    torch.save(model.state_dict(), f"stage1_epoch_{epoch+1}.pth")

Epoch 1/12: 100%|██████████| 6250/6250 [2:02:23<00:00,  1.18s/it, loss=0.0892, train_acc=0.967]   



Epoch 1 | Train Acc: 0.9669 | Val Acc: 0.7603


Epoch 2/12: 100%|██████████| 6250/6250 [1:38:58<00:00,  1.05it/s, loss=0.00309, train_acc=0.987] 



Epoch 2 | Train Acc: 0.9873 | Val Acc: 0.7332


Epoch 3/12: 100%|██████████| 6250/6250 [1:36:09<00:00,  1.08it/s, loss=0.00171, train_acc=0.992] 



Epoch 3 | Train Acc: 0.9916 | Val Acc: 0.8249


Epoch 4/12:  10%|█         | 642/6250 [09:01<1:18:50,  1.19it/s, loss=0.00288, train_acc=0.993] 


KeyboardInterrupt: 